In [4]:
import pandas as pd
from sqlalchemy import types, create_engine
from sqlalchemy.engine.url import URL
import psycopg2
from sqlalchemy.orm import sessionmaker
import pyodbc
import urllib
import json
from sqlalchemy import text

In [5]:
def to_postgres(user,server,database):
    engine_postgres = create_engine(f'postgresql+psycopg2://postgres:{user}@{server}/{database}')
    return engine_postgres

In [6]:
def csv_to_df(file_path):
    df = pd.read_csv(file_path)
    return df

In [7]:
def df_to_csv(df, file_path):
    df.to_csv(file_path, index=False)

In [8]:
def raw_csv_processing():
    source_path = 'data source'
    destination_path = 'data target/raw'
    list_of_file = ['dummy_realisasi_bus.csv','dummy_routes.csv','dummy_shelter_corridor.csv']
    for file in list_of_file:
        df = csv_to_df(f'{source_path}/{file}')
        df_to_csv(df, f'{destination_path}/{file}')
        print(f"File {file} has been processed.")
    print("All files have been processed.")


In [9]:
def raw_csv_to_postgres():
    source_path = 'data source'
    pg_server = 'localhost'
    pg_database = 'transjakarta'
    pg_schema = 'raw'
    pg_user = 'postgres'
    engine_postgres = to_postgres(pg_user,pg_server,pg_database)
    list_of_file = ['dummy_transaksi_bus.csv','dummy_transaksi_halte.csv']
    for file in list_of_file:
        df = csv_to_df(f'{source_path}/{file}')
        table_name = file.replace('dummy_','').replace('.csv','')
        df.to_sql(table_name, engine_postgres, if_exists='replace', index=False, schema=pg_schema)
        print(f"File {file} has been uploaded to PostgreSQL as table {table_name}.")
    print("All files have been uploaded to PostgreSQL.")

In [10]:
from sqlalchemy import text

def stage_csv_to_postgres():
    pg_procedure = [
        'insert_to_transaksi_bus()',
        'insert_to_transaksi_halte()'
    ]

    pg_server = 'localhost'
    pg_database = 'transjakarta'
    pg_user = 'postgres'
    pg_schema = 'stage'

    engine_postgres = to_postgres(pg_user, pg_server, pg_database)

    with engine_postgres.connect() as connection:
        connection.execute(text(f'SET search_path TO {pg_schema};'))

        for procedure in pg_procedure:
            sql = text(f'CALL {procedure};')
            connection.execute(sql)
            print(f"Procedure {procedure} has been executed.")

    print("All procedures have been executed.")

In [11]:
from sqlalchemy import text

def target_csv_to_postgres():
    pg_procedure = [
        'merge_transaksi_bus()',
        'merge_transaksi_halte()'
    ]

    pg_server = 'localhost'
    pg_database = 'transjakarta'
    pg_user = 'postgres'
    pg_schema = 'target'

    engine_postgres = to_postgres(pg_user, pg_server, pg_database)

    with engine_postgres.connect() as connection:
        connection.execute(text(f'SET search_path TO {pg_schema};'))

        for procedure in pg_procedure:
            sql = text(f'CALL {procedure};')
            connection.execute(sql)
            print(f"Procedure {procedure} has been executed.")

    print("All procedures have been executed.")

In [12]:
def stage_csv_to_csv():
    file_map = {
        "data target/raw/dummy_realisasi_bus.csv": "data target/stage/realisasi_bus.csv",
        "data target/raw/dummy_routes.csv": "data target/stage/routes.csv",
        "data target/raw/dummy_shelter_corridor.csv": "data target/stage/shelter_corridor.csv"
    }

    for src, dst in file_map.items():
        df = pd.read_csv(src)

        # Special case: only for realisasi_bus
        if "realisasi_bus" in src:
            # FIX: %M = minutes, %m = month
            df["tanggal_realisasi"] = pd.to_datetime(
                df["tanggal_realisasi"],
                format="%d/%m/%Y",
                errors="coerce"  # prevent crash on invalid dates
            )

        df_to_csv(df, dst)
        print(f"Processed: {src} -> {dst}")
    print("All files have been processed.")

In [13]:
import pandas as pd
import json
import os

def load_config():
    with open("data target/config/table_config.json", "r") as f:
        return json.load(f)

def merge_by_pk(df_stage, df_target, pk):
    if df_target is None:
        return df_stage

    df_merged = pd.concat([df_target, df_stage], ignore_index=True)
    df_merged = df_merged.drop_duplicates(subset=[pk], keep="last")
    return df_merged

def process_table(name, cfg):
    print(f"Processing: {name}")

    # Read source CSV
    df = pd.read_csv(cfg["source"])

    # Apply date parsing
    if "date_columns" in cfg:
        for col, fmt in cfg["date_columns"].items():
            df[col] = pd.to_datetime(df[col], format=fmt)

    # Write to stage
    df.to_csv(cfg["stage"], index=False)

    # Load target (if exists)
    df_target = pd.read_csv(cfg["target"]) if os.path.exists(cfg["target"]) else None

    # Merge stage → target
    df_merged = merge_by_pk(df, df_target, cfg["pk"])

    # Write to target
    df_merged.to_csv(cfg["target"], index=False)

    print(f"✔ Completed {name}\n")

def target_csv_to_csv():
    config = load_config()
    for name, cfg in config.items():
        process_table(name, cfg)


In [14]:
realisasi_bus = pd.read_csv('data target/raw/dummy_realisasi_bus.csv')

In [15]:
routes = pd.read_csv('data target/raw/dummy_routes.csv')

In [16]:
shelter_corridor = pd.read_csv('data target/raw/dummy_shelter_corridor.csv')

In [17]:
realisasi_bus.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 515 entries, 0 to 514
Data columns (total 3 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   tanggal_realisasi  515 non-null    object
 1   bus_body_no        515 non-null    object
 2   rute_realisasi     515 non-null    object
dtypes: object(3)
memory usage: 12.2+ KB


In [18]:
routes.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21 entries, 0 to 20
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   route_code  21 non-null     object
 1   route_name  21 non-null     object
dtypes: object(2)
memory usage: 464.0+ bytes


In [19]:
shelter_corridor.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 74 entries, 0 to 73
Data columns (total 3 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   shelter_name_var  74 non-null     object
 1   corridor_code     74 non-null     int64 
 2   corridor_name     74 non-null     object
dtypes: int64(1), object(2)
memory usage: 1.9+ KB


In [20]:
raw_csv_processing()

File dummy_realisasi_bus.csv has been processed.
File dummy_routes.csv has been processed.
File dummy_shelter_corridor.csv has been processed.
All files have been processed.


In [21]:
raw_csv_to_postgres()

File dummy_transaksi_bus.csv has been uploaded to PostgreSQL as table transaksi_bus.
File dummy_transaksi_halte.csv has been uploaded to PostgreSQL as table transaksi_halte.
All files have been uploaded to PostgreSQL.


In [22]:
stage_csv_to_postgres()

Procedure insert_to_transaksi_bus() has been executed.
Procedure insert_to_transaksi_halte() has been executed.
All procedures have been executed.


In [23]:
target_csv_to_postgres()

Procedure merge_transaksi_bus() has been executed.
Procedure merge_transaksi_halte() has been executed.
All procedures have been executed.


In [24]:
stage_csv_to_csv()

Processed: data target/raw/dummy_realisasi_bus.csv -> data target/stage/realisasi_bus.csv
Processed: data target/raw/dummy_routes.csv -> data target/stage/routes.csv
Processed: data target/raw/dummy_shelter_corridor.csv -> data target/stage/shelter_corridor.csv
All files have been processed.


In [25]:
target_csv_to_csv()

Processing: realisasi_bus
✔ Completed realisasi_bus

Processing: routes
✔ Completed routes

Processing: shelter_corridor
✔ Completed shelter_corridor



In [27]:
bus_transaction = pd.read_sql('SELECT * FROM stage.transaksi_bus', con=to_postgres('postgres','localhost','transjakarta'))
bus_transaction.head()

,uuid,waktu_transaksi,armada_id_var,no_body_var,card_number_var,card_type_var,balance_before_int,fare_int,balance_after_int,transcode_txt,gate_in_boo,p_latitude_flo,p_longitude_flo,status_var,free_service_boo,insert_on_dtm,dw_load_ts
0,40ffc2dc-933c-4e21-ab20-3935e33deac0,2025-07-30 02:18:36,B 4738 SEL,KLG-459,326553144400708,BRIZZI,54000,35000,19000,TX000001,True,-6.192278,106.778723,S,False,2025-07-30 02:20:17,2025-11-15 13:00:34.957595
1,cc03bbdf-e9da-457d-874a-a4ab944081e2,2025-07-26 08:52:57,B 5429 PKC,LGS-431,9673319555070388,E-Money,86000,35000,51000,TX000002,True,-6.240312,106.781831,S,False,2025-07-26 08:54:06,2025-11-15 13:00:34.957595
2,d146c50b-d3ed-4029-b0fe-ee072736b4fb,2025-07-15 07:06:30,B 2169 GFP,KLG-191,8705513299172500,BRIZZI,74000,35000,39000,TX000003,False,-6.189989,106.762906,S,False,2025-07-15 07:06:35,2025-11-15 13:00:34.957595
3,6972e293-1c2f-424d-b6e0-e1a65420bd5a,2025-07-14 22:47:38,B 6727 BTH,KLG-664,5737754300732352,JakCard,47500,3500,44000,TX000004,False,-6.152841,106.758245,S,False,2025-07-14 22:48:36,2025-11-15 13:00:34.957595
4,220d052f-bce3-4969-b076-e38820921cb8,2025-07-07 11:20:32,B 7517 GGE,NBR-758,9731046849153976,Flazz,98500,3500,95000,TX000005,True,-6.160012,106.813966,S,False,2025-07-07 11:21:09,2025-11-15 13:00:34.957595
